# Full OCR Preprocessing Pipeline + CER Evaluation

In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null!pip install -q pytesseract opencv-python-headless pillow numpy

In [ ]:
import reimport cv2import numpy as npimport pytesseractfrom PIL import Image, ImageDraw, ImageFont

In [ ]:
DATASET_PATH = ""GROUND_TRUTH_TEXT = "Tesseract OCR converts images of text into machine readable text."TESSERACT_CONFIG = "--oem 3 --psm 6"

## Sample Image

In [ ]:
def create_sample_image(text, width=900, height=220, font_size=28, angle=4, noise=True):    image = Image.new("RGB", (width, height), color=(255, 255, 255))    draw = ImageDraw.Draw(image)    try:        font = ImageFont.truetype("DejaVuSans.ttf", font_size)    except IOError:        font = ImageFont.load_default()    draw.text((20, height // 2 - font_size), text, fill=(0, 0, 0), font=font)    image = image.rotate(angle, fillcolor=(255, 255, 255), expand=False)    if noise:        arr = np.array(image).astype(np.int16)        gaussian_noise = np.random.normal(0, 12, arr.shape).astype(np.int16)        arr = np.clip(arr + gaussian_noise, 0, 255).astype(np.uint8)        image = Image.fromarray(arr)    return imagesample_image = create_sample_image(GROUND_TRUTH_TEXT)sample_image

In [ ]:
def load_image(dataset_path, fallback_image):    if dataset_path:        return np.array(Image.open(dataset_path).convert("RGB"))    return np.array(fallback_image)image_bgr = cv2.cvtColor(load_image(DATASET_PATH, sample_image), cv2.COLOR_RGB2BGR)

## Grayscale Conversion

In [ ]:
def to_grayscale(image_bgr):    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)gray = to_grayscale(image_bgr)Image.fromarray(gray)

## Denoising — Gaussian and Median Filters

In [ ]:
def denoise_gaussian(gray, kernel_size=5, sigma=0):    return cv2.GaussianBlur(gray, (kernel_size, kernel_size), sigma)def denoise_median(gray, kernel_size=3):    return cv2.medianBlur(gray, kernel_size)gaussian_denoised = denoise_gaussian(gray)median_denoised = denoise_median(gray)Image.fromarray(np.hstack([gaussian_denoised, median_denoised]))

## Deskew via Hough Transform

In [ ]:
def compute_skew_angle_hough(gray, canny_low=50, canny_high=150, hough_threshold=80, min_line_length=100, max_line_gap=10):    edges = cv2.Canny(gray, canny_low, canny_high, apertureSize=3)    lines = cv2.HoughLinesP(        edges, 1, np.pi / 180, threshold=hough_threshold,        minLineLength=min_line_length, maxLineGap=max_line_gap    )    if lines is None:        return 0.0    angles = []    for line in lines:        x1, y1, x2, y2 = line[0]        angle = np.degrees(np.arctan2(y2 - y1, x2 - x1))        if abs(angle) < 45:            angles.append(angle)    if len(angles) == 0:        return 0.0    return float(np.median(angles))def deskew_hough(image, gray_for_angle=None):    reference = gray_for_angle if gray_for_angle is not None else image    angle = compute_skew_angle_hough(reference)    h, w = image.shape[:2]    center = (w // 2, h // 2)    rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)    rotated = cv2.warpAffine(        image, rotation_matrix, (w, h),        flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE    )    return rotated, angledeskewed, detected_angle = deskew_hough(median_denoised)print("detected skew angle:", detected_angle)Image.fromarray(deskewed)

## Binarization — Otsu and Adaptive Thresholding

In [ ]:
def binarize_otsu(gray):    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)    return threshdef binarize_adaptive_mean(gray, block_size=25, c=10):    return cv2.adaptiveThreshold(        gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, block_size, c    )def binarize_adaptive_gaussian(gray, block_size=25, c=10):    return cv2.adaptiveThreshold(        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, block_size, c    )otsu_bin = binarize_otsu(deskewed)adaptive_mean_bin = binarize_adaptive_mean(deskewed)adaptive_gaussian_bin = binarize_adaptive_gaussian(deskewed)Image.fromarray(np.hstack([otsu_bin, adaptive_mean_bin, adaptive_gaussian_bin]))

## Morphological Operations — Erosion, Dilation, Opening, Closing

In [ ]:
def erode(binary, kernel_size=2, iterations=1):    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_size, kernel_size))    return cv2.erode(binary, kernel, iterations=iterations)def dilate(binary, kernel_size=2, iterations=1):    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_size, kernel_size))    return cv2.dilate(binary, kernel, iterations=iterations)def opening(binary, kernel_size=2):    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_size, kernel_size))    return cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)def closing(binary, kernel_size=2):    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_size, kernel_size))    return cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)inverted = cv2.bitwise_not(otsu_bin)eroded = erode(inverted)dilated = dilate(inverted)opened = opening(inverted)closed = closing(inverted)Image.fromarray(cv2.bitwise_not(np.hstack([eroded, dilated, opened, closed])))

## Full Preprocessing Pipeline

In [ ]:
def preprocess_image(    image_bgr,    denoise_method="gaussian",    binarize_method="otsu",    morph_op="closing",    gaussian_kernel=5,    median_kernel=3,    adaptive_block_size=25,    adaptive_c=10,    morph_kernel_size=2,):    gray = to_grayscale(image_bgr)    if denoise_method == "gaussian":        denoised = denoise_gaussian(gray, gaussian_kernel)    elif denoise_method == "median":        denoised = denoise_median(gray, median_kernel)    else:        denoised = gray    deskewed, angle = deskew_hough(denoised)    if binarize_method == "otsu":        binary = binarize_otsu(deskewed)    elif binarize_method == "adaptive_mean":        binary = binarize_adaptive_mean(deskewed, adaptive_block_size, adaptive_c)    elif binarize_method == "adaptive_gaussian":        binary = binarize_adaptive_gaussian(deskewed, adaptive_block_size, adaptive_c)    else:        binary = deskewed    inverted = cv2.bitwise_not(binary)    if morph_op == "erode":        morphed = erode(inverted, morph_kernel_size)    elif morph_op == "dilate":        morphed = dilate(inverted, morph_kernel_size)    elif morph_op == "opening":        morphed = opening(inverted, morph_kernel_size)    elif morph_op == "closing":        morphed = closing(inverted, morph_kernel_size)    else:        morphed = inverted    final = cv2.bitwise_not(morphed)    return final, angleprocessed_image, skew_angle = preprocess_image(image_bgr)print("detected skew angle:", skew_angle)Image.fromarray(processed_image)

## OCR on Raw vs Preprocessed Image

In [ ]:
raw_text = pytesseract.image_to_string(image_bgr, config=TESSERACT_CONFIG).strip()processed_text = pytesseract.image_to_string(processed_image, config=TESSERACT_CONFIG).strip()print("raw ocr text:", raw_text)print("preprocessed ocr text:", processed_text)

## CER Evaluation

In [ ]:
def normalize_text(text, case_sensitive=False, keep_chars=None):    if not case_sensitive:        text = text.lower()    if keep_chars is not None:        pattern = f"[^{re.escape(keep_chars)}]"        text = re.sub(pattern, "", text)    text = re.sub(r"\s+", " ", text).strip()    return list(text)

In [ ]:
def edit_distance_ops(ref_chars, hyp_chars):    n, m = len(ref_chars), len(hyp_chars)    dp = [[0] * (m + 1) for _ in range(n + 1)]    for i in range(n + 1):        dp[i][0] = i    for j in range(m + 1):        dp[0][j] = j    for i in range(1, n + 1):        for j in range(1, m + 1):            if ref_chars[i - 1] == hyp_chars[j - 1]:                dp[i][j] = dp[i - 1][j - 1]            else:                substitution = dp[i - 1][j - 1] + 1                insertion = dp[i][j - 1] + 1                deletion = dp[i - 1][j] + 1                dp[i][j] = min(substitution, insertion, deletion)    i, j = n, m    substitutions = insertions = deletions = 0    while i > 0 or j > 0:        if i > 0 and j > 0 and ref_chars[i - 1] == hyp_chars[j - 1]:            i -= 1            j -= 1        elif i > 0 and j > 0 and dp[i][j] == dp[i - 1][j - 1] + 1:            substitutions += 1            i -= 1            j -= 1        elif j > 0 and dp[i][j] == dp[i][j - 1] + 1:            insertions += 1            j -= 1        else:            deletions += 1            i -= 1    return substitutions, deletions, insertions, dp[n][m]

In [ ]:
def cer(reference, hypothesis, case_sensitive=False, keep_chars=None):    ref_chars = normalize_text(reference, case_sensitive, keep_chars)    hyp_chars = normalize_text(hypothesis, case_sensitive, keep_chars)    substitutions, deletions, insertions, _ = edit_distance_ops(ref_chars, hyp_chars)    n = len(ref_chars) if len(ref_chars) > 0 else 1    return (substitutions + deletions + insertions) / n

In [ ]:
def batch_cer(pairs, case_sensitive=False, keep_chars=None):    total_s = total_d = total_i = total_n = 0    per_example = []    for reference, hypothesis in pairs:        ref_chars = normalize_text(reference, case_sensitive, keep_chars)        hyp_chars = normalize_text(hypothesis, case_sensitive, keep_chars)        s, d, i, _ = edit_distance_ops(ref_chars, hyp_chars)        n = len(ref_chars) if len(ref_chars) > 0 else 1        example_cer = (s + d + i) / n        per_example.append(example_cer)        total_s += s        total_d += d        total_i += i        total_n += len(ref_chars)    corpus_n = total_n if total_n > 0 else 1    corpus_cer = (total_s + total_d + total_i) / corpus_n    return {        "per_example_cer": per_example,        "corpus_cer": corpus_cer,        "substitutions": total_s,        "deletions": total_d,        "insertions": total_i,        "total_chars": total_n    }

In [ ]:
pairs = [    (GROUND_TRUTH_TEXT, raw_text),    (GROUND_TRUTH_TEXT, processed_text),]results = batch_cer(pairs)labels = ["raw", "preprocessed"]for label, (reference, hypothesis), example_cer in zip(labels, pairs, results["per_example_cer"]):    print(f"[{label}]")    print(f"ref: {reference}")    print(f"hyp: {hypothesis}")    print(f"cer: {example_cer:.3f}")    print("-" * 40)print(f"corpus cer: {results['corpus_cer']:.3f}")print(f"substitutions: {results['substitutions']}")print(f"deletions: {results['deletions']}")print(f"insertions: {results['insertions']}")print(f"total reference chars: {results['total_chars']}")